# Poglavlje 7: Izrada chat aplikacija
## Brzi početak rada s OpenAI API-jem

Ovaj bilježnik je prilagođen iz [Azure OpenAI Primjeraka Repozitorija](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) koji uključuje bilježnike koji pristupaju [Azure OpenAI](aoai-assignment.ipynb) uslugama.

Python OpenAI API također radi s Azure OpenAI modelima, uz nekoliko prilagodbi. Saznajte više o razlikama ovdje: [Kako prebacivati između OpenAI i Azure OpenAI krajnjih točaka s Python-om](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Pregled  
"Veliki modeli jezika su funkcije koje preslikavaju tekst u tekst. S obzirom na ulazni niz teksta, veliki model jezika pokušava predvidjeti tekst koji će slijediti"(1). Ovaj "quickstart" bilježnik će korisnike upoznati s osnovnim konceptima velikih jezičnih modela, glavnim paketnim zahtjevima za početak rada s AML-om, laganim uvodom u dizajn upita (promptova) i nekoliko kratkih primjera različitih primjena. 


## Sadržaj  

[Pregled](#overview)  
[Kako koristiti OpenAI uslugu](#how-to-use-openai-service)  
[1. Kreiranje vaše OpenAI usluge](#1.-creating-your-openai-service)  
[2. Instalacija](#2.-installation)    
[3. Podaci za prijavu](#3.-credentials)  

[Primjeri korištenja](#use-cases)    
[1. Sažimanje teksta](#1.-summarize-text)  
[2. Klasifikacija teksta](#2.-classify-text)  
[3. Generiranje novih naziva proizvoda](#3.-generate-new-product-names)  
[4. Fino podešavanje klasifikatora](#4.fine-tune-a-classifier)  

[Reference](#references)


### Napravite svoj prvi prompt  
Ova kratka vježba pružit će osnovni uvod za slanje promptova OpenAI modelu za jednostavan zadatak "sažimanje".


**Koraci**:  
1. Instalirajte OpenAI biblioteku u vaše python okruženje  
2. Učitajte standardne pomoćne biblioteke i postavite svoje uobičajene sigurnosne vjerodajnice za OpenAI servis koji ste kreirali  
3. Odaberite model za svoj zadatak  
4. Napravite jednostavan prompt za model  
5. Pošaljite svoj zahtjev model API-ju!


### 1. Instalirajte OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Uvezi pomoćne biblioteke i stvori vjerodajnice


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Pronalaženje pravog modela  
Modele poput GPT-4o i GPT-4o mini mogu razumjeti i generirati prirodni jezik, i dostupni su na OpenAI platformi s različitim razinama snage i brzine prikladnim za različite zadatke.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Dizajn upita  

"Čarolija velikih jezičnih modela je u tome što se treniranjem na minimiziranju ove pogreške predviđanja preko ogromnih količina teksta modeli nauče pojmove korisne za ta predviđanja. Na primjer, nauče pojmove poput"(1):

* kako se piše
* kako funkcionira gramatika
* kako parafrazirati
* kako odgovarati na pitanja
* kako voditi razgovor
* kako pisati na mnogim jezicima
* kako kodirati
* itd.

#### Kako kontrolirati veliki jezični model  
"Od svih ulaza u veliki jezični model, daleko najutjecajniji je tekstualni upit(1).

Velike jezične modele možete potaknuti da proizvedu izlaz na nekoliko načina:

Instrukcija: Recite modelu što želite
Završetak: Navedite model da dovrši početak onoga što želite
Demonstracija: Pokažite modelu što želite, s pomoću:
Nekoliko primjera u upitu
Mnoge stotine ili tisuće primjera u skupu podataka za fino podešavanje"



#### Postoje tri osnovna pravila za stvaranje upita:

**Pokaži i reci**. Jasno naznačite što želite bilo kroz upute, primjere ili kombinaciju oboje. Ako želite da model rangira popis stavki po abecedi ili klasificira odlomak po osjećaju, pokažite mu da je to ono što želite.

**Osigurajte kvalitetne podatke**. Ako pokušavate izgraditi klasifikator ili natjerati model da slijedi uzorak, pazite da ima dovoljno primjera. Pažljivo pregledajte svoje primjere — model je obično dovoljno pametan da može uočiti osnovne pravopisne pogreške i dati odgovor, ali također može pretpostaviti da je to namjerno što može utjecati na odgovor.

**Provjerite svoje postavke.** Postavke temperature i top_p kontroliraju koliko je model deterministički u generiranju odgovora. Ako tražite odgovor gdje postoji samo jedan točan odgovor, postavite ih niže. Ako želite raznovrsnije odgovore, postavite ih više. Najčešća pogreška ljudi s ovim postavkama je što pogrešno misle da su to kontrole "pameti" ili "kreativnosti".


Izvor: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Pošalji!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ponovite isti poziv, kako se rezultati uspoređuju?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Sažmi tekst  
#### Izazov  
Sažmi tekst dodavanjem 'tl;dr:' na kraj tekstualnog dijela. Primijetite kako model razumije kako obaviti niz zadataka bez dodatnih uputa. Možete eksperimentirati s opisnijim uputama od tl;dr kako biste promijenili ponašanje modela i prilagodili sažimanje koje dobivate(3).  

Nedavni radovi pokazali su znatna poboljšanja na mnogim NLP zadacima i mjernim ljestvicama kroz predtreniranje na velikom korpusu teksta, praćeno fino podešavanje na određenom zadatku. Iako je arhitektura obično neovisna o zadatku, ova metoda još uvijek zahtijeva zadatku specifične skupove za fino podešavanje od tisuća ili desetaka tisuća primjera. S druge strane, ljudi općenito mogu obaviti novi jezični zadatak samo s nekoliko primjera ili jednostavnim uputama – nešto što trenutni NLP sustavi uglavnom još uvijek nisu u stanju učiniti. Ovdje pokazujemo da skaliranje jezičnih modela znatno poboljšava sposobnost obavljanja zadataka s malo primjera, ponekad čak dosežući konkurentnost s prethodnim najnaprednijim metodama finog podešavanja.  



Tl;dr


# Vježbe za nekoliko slučajeva upotrebe  
1. Sažetak teksta  
2. Klasifikacija teksta  
3. Generiranje novih naziva proizvoda


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasificiraj tekst  
#### Izazov  
Klasificirajte stavke u kategorije koje su pružene u trenutku izvođenja. U sljedećem primjeru, u prompt (*playground_reference) dajemo i kategorije i tekst za klasifikaciju. 

Upit kupca: Bok, jedan od tipki na tipkovnici mog prijenosnog računala nedavno se pokvario i trebat će mi zamjena:

Klasificirana kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generirajte nove nazive proizvoda
#### Izazov
Izradite nazive proizvoda na temelju primjera riječi. Ovdje uključujemo u upit informacije o proizvodu za koji ćemo generirati nazive. Također pružamo sličan primjer kako bismo pokazali obrazac koji želimo dobiti. Također smo postavili visoku vrijednost temperature kako bismo povećali slučajnost i dobili inovativnije odgovore.

Opis proizvoda: Kućni proizvođač milkshakea
Riječi za inspiraciju: brz, zdrav, kompaktan.
Nazivi proizvoda: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis proizvoda: Par cipela koji može odgovarati na svaki broj stopala.
Riječi za inspiraciju: prilagodljiv, odgovara, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najbolje prakse za fino podešavanje GPT modela za klasifikaciju teksta](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za Više Pomoći  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Suradnici
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
